In [23]:
import yaml
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [24]:
with open("D:\Github Sanity\ML Garden\projects\pistachio_prediction\config.yaml", "r") as file:
    config_data = yaml.safe_load(file)

In [25]:
data = pd.read_csv(config_data["dataset_path"])
data

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,ROUNDNESS,COMPACTNESS,SHAPEFACTOR_1,SHAPEFACTOR_2,SHAPEFACTOR_3,SHAPEFACTOR_4,Class
0,73107,1161.8070,442.4074,217.7261,0.8705,305.0946,0.9424,77579,0.7710,2.0319,0.6806,0.6896,0.0061,0.0030,0.4756,0.9664,Kirmizi_Pistachio
1,89272,1173.1810,460.2551,251.9546,0.8369,337.1419,0.9641,92598,0.7584,1.8267,0.8151,0.7325,0.0052,0.0028,0.5366,0.9802,Siit_Pistachio
2,60955,999.7890,386.9247,209.1255,0.8414,278.5863,0.9465,64400,0.7263,1.8502,0.7663,0.7200,0.0063,0.0034,0.5184,0.9591,Kirmizi_Pistachio
3,79537,1439.5129,466.7973,221.2136,0.8806,318.2289,0.9437,84281,0.7568,2.1102,0.4823,0.6817,0.0059,0.0028,0.4648,0.9807,Kirmizi_Pistachio
4,96395,1352.6740,515.8730,246.5945,0.8784,350.3340,0.9549,100950,0.7428,2.0920,0.6620,0.6791,0.0054,0.0026,0.4612,0.9648,Kirmizi_Pistachio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1713,65570,2071.4451,418.0258,217.1458,0.8545,288.9400,0.8976,73054,0.5945,1.9251,0.1920,0.6912,0.0064,0.0033,0.4778,0.9197,Kirmizi_Pistachio
1714,68849,1441.2590,451.0457,205.2553,0.8905,296.0764,0.9340,73716,0.6459,2.1975,0.4165,0.6564,0.0066,0.0030,0.4309,0.9469,Kirmizi_Pistachio
1715,90270,1370.5380,428.9636,269.8232,0.7774,339.0211,0.9722,92847,0.7400,1.5898,0.6039,0.7903,0.0048,0.0030,0.6246,0.9930,Siit_Pistachio
1716,73148,1309.8430,469.0491,208.3141,0.8960,305.1801,0.9376,78014,0.6341,2.2516,0.5358,0.6506,0.0064,0.0028,0.4233,0.9532,Kirmizi_Pistachio


In [26]:
X = data[config_data["independent_features"]]
y = data[config_data["dependent_features"]]

In [27]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

C:\Users\swana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [28]:
selector = SelectKBest(score_func=f_classif, k=10)  # Select top 10 features
X_new = selector.fit_transform(X, y)

# Save selected feature indices for reference
selected_features = selector.get_support(indices=True)
print(f"Selected Features (Indices): {selected_features}")

Selected Features (Indices): [ 0  1  3  4  5  7  9 11 12 14]


In [29]:
columns = config_data["independent_features"]
for i in selected_features:
    print(columns[i])

AREA
PERIMETER
MINOR_AXIS
ECCENTRICITY
EQDIASQ
CONVEX_AREA
ASPECT_RATIO
COMPACTNESS
SHAPEFACTOR_1
SHAPEFACTOR_3


In [30]:
scaler = StandardScaler()
X_new = scaler.fit_transform(X_new)

In [31]:
rf = RandomForestClassifier(n_estimators=50, random_state=42)
ada = AdaBoostClassifier(n_estimators=50, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

# Create Voting Classifier (Hard Voting)
voting_classifier = VotingClassifier(
    estimators=[('rf', rf), ('ada', ada), ('knn', knn)],
    voting='hard'  # Use 'soft' if models support predict_proba()
)

# Train the ensemble model
voting_classifier.fit(X_new, y)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(n_estimators=50,
                                                     random_state=42)),
                             ('ada', AdaBoostClassifier(random_state=42)),
                             ('knn', KNeighborsClassifier())])

In [32]:
sample = [[60955,999.789,386.9247,209.1255,0.8414,278.5863,0.9465,64400,0.7263,1.8502,0.7663,0.72,0.0063,0.0034,0.5184,0.9591]]
sample_new = scaler.transform(selector.transform(sample))
predicted_label = voting_classifier.predict(sample_new)
print(f"Predicted Label: {encoder.inverse_transform(predicted_label)}")

Predicted Label: ['Kirmizi_Pistachio']


C:\Users\swana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SelectKBest was fitted with feature names
  warnings.warn(


In [33]:
y_pred = voting_classifier.predict(X_new)

In [34]:
cm = confusion_matrix(y, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y, y_pred))

Confusion Matrix:
[[936  62]
 [ 84 636]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       998
           1       0.91      0.88      0.90       720

    accuracy                           0.92      1718
   macro avg       0.91      0.91      0.91      1718
weighted avg       0.91      0.92      0.91      1718

